<a href="https://colab.research.google.com/github/AngellyC07/Ciencia_de_datos/blob/main/Censo_poblacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Estructura general del trabajo
Agenda
1. Cargar el dataset
2. Inspección inicial del archivo
3. Construcción del diccionario de datos
4. Clasificación de columnas según su utilidad
5. Diagnóstico de valores nulos
6. Tratamiento de valores faltantes sin borrar registros
7. Detección y corrección de datos mal colocados
8. Estandarización de tipos de datos
9. Tabla de decisiones: columnas a conservar, revisar o excluir
10. Base limpia para EDA
11. EDA inicial del dataset limpio

In [ ]:
# ==========================================
# PASO 1. CARGA DE LIBRERÍAS Y DEL ARCHIVO
# ==========================================

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Configuración visual
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Ruta del archivo en Colab
archivo = "/content/CENSO POBLACIONAL AGOSTO.xlsx"

# Ver hojas disponibles
xls = pd.ExcelFile(archivo)
print("Hojas disponibles:")
print(xls.sheet_names)

# Cargar la primera hoja
df = pd.read_excel(archivo, sheet_name=xls.sheet_names[0])

# Crear copia de seguridad y copia de trabajo
df_raw = df.copy()
df_clean = df.copy()

print("\nDimensiones del dataset:")
print(df_clean.shape)

print("\nPrimeras 5 filas:")
display(df_clean.head())

In [ ]:
# ==========================================
# PASO 2. INSPECCIÓN INICIAL
# ==========================================

print("Información general del dataset:")
print(df_clean.info())

print("\nNúmero de filas y columnas:")
print(df_clean.shape)

print("\nNombres de las columnas:")
display(pd.DataFrame({"Columnas": df_clean.columns}))

In [ ]:
# ==========================================
# PASO 3. DICCIONARIO DE DATOS INICIAL
# ==========================================

def detectar_rol_variable(col):
    c = col.lower()

    if c.startswith("ide_") or c.startswith("cod_") or c in ["num_documento", "num_ficha", "num_paquete"]:
        return "Identificador / código"
    elif c.startswith("fec_"):
        return "Fecha"
    elif c.startswith("ind_"):
        return "Indicador / binaria"
    elif c.startswith("vlr_"):
        return "Valor monetario"
    elif c.startswith("num_"):
        return "Conteo / cantidad"
    elif "nom_" in c or "nombre" in c or "apellido" in c or "dir_" in c or "email" in c:
        return "Texto descriptivo"
    else:
        return "Variable general"

diccionario = pd.DataFrame({
    "Variable": df_clean.columns,
    "Tipo_dato_actual": df_clean.dtypes.astype(str).values,
    "No_nulos": df_clean.notnull().sum().values,
    "Nulos": df_clean.isnull().sum().values,
    "%_nulos": (df_clean.isnull().mean() * 100).round(2).values,
    "Valores_unicos": df_clean.nunique(dropna=True).values,
    "Rol_preliminar": [detectar_rol_variable(col) for col in df_clean.columns]
})

# Agregar ejemplos de valores reales
ejemplos = []
for col in df_clean.columns:
    vals = df_clean[col].dropna().astype(str).unique()[:3]
    ejemplos.append(" | ".join(vals))

diccionario["Ejemplos_valores"] = ejemplos

display(diccionario.head(20))